<a href="https://www.kaggle.com/code/codeantara093/redrob-candidate-dashboard-1?scriptVersionId=331507605" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
# ── SECTION 1: Setup & Dependencies ──────────────────────────────────────────
!pip install -q FlagEmbedding rank_bm25 python-docx

import json, re, math, os, warnings, shutil
import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
from tqdm.auto import tqdm
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
import spacy
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ── Kaggle paths ──────────────────────────────────────────────────────────────
DATA_DIR   = "/kaggle/input/datasets/codeantara093/india-runs"
OUTPUT_DIR = "/kaggle/working"
EMB_CACHE  = "/kaggle/working/cand_embs.npy"

CAND_FILE    = f"{DATA_DIR}/candidates.jsonl"       # 100k test candidates
SAMPLE_FILE  = f"{DATA_DIR}/sample_candidates.json"  # 50 training candidates
JD_FILE      = f"{DATA_DIR}/job_description.docx"
VALIDATE_PY  = f"{DATA_DIR}/validate_submission.py"

# ── Updated constants for Top 100 ────────────────────────────────────────────
TOP_K_RETRIEVAL = 300   # wider net → more candidates for final 100
TOP_K_RERANK    = 150   # reranker scores top 150
TOP_K_FINAL     = 100   # final output = Top 100
BM25_WEIGHT     = 0.35
EMB_WEIGHT      = 0.65

nlp = spacy.load("en_core_web_sm")

# Verify files exist
for path in [CAND_FILE, SAMPLE_FILE, JD_FILE]:
    status = "✅" if os.path.exists(path) else "❌"
    print(f"{status} {path}")

print(f"\n✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"✅ TOP_K_FINAL = {TOP_K_FINAL}")

✅ /kaggle/input/datasets/codeantara093/india-runs/candidates.jsonl
✅ /kaggle/input/datasets/codeantara093/india-runs/sample_candidates.json
✅ /kaggle/input/datasets/codeantara093/india-runs/job_description.docx

✅ GPU: Tesla T4
✅ TOP_K_FINAL = 100


In [3]:
# ── SECTION 2: Load All Data Files ───────────────────────────────────────────

# 1. Load 100k test candidates
candidates = []
with open(CAND_FILE) as f:
    for line in f:
        line = line.strip()
        if line: candidates.append(json.loads(line))
print(f"✅ Test candidates: {len(candidates):,}")

# 2. Load 50 sample (training) candidates
with open(SAMPLE_FILE) as f:
    sample_data = json.load(f)

# Handle both list and dict formats
if isinstance(sample_data, list):
    sample_candidates = sample_data
elif isinstance(sample_data, dict):
    # might be {"candidates": [...]} or {"data": [...]}
    sample_candidates = (sample_data.get("candidates") or
                         sample_data.get("data") or
                         list(sample_data.values())[0])

print(f"✅ Sample (train) candidates: {len(sample_candidates)}")
print(f"📋 Sample keys: {list(sample_candidates[0].keys())}")

# Check if sample has relevance/score labels
sample_keys = set(sample_candidates[0].keys())
LABEL_FIELD = None
for possible in ["relevance_score", "score", "label",
                  "relevance", "rank", "rating"]:
    if possible in sample_keys:
        LABEL_FIELD = possible
        break

if LABEL_FIELD:
    print(f"✅ Found label field: '{LABEL_FIELD}'")
    labels = [c.get(LABEL_FIELD, 0) for c in sample_candidates]
    print(f"   Label range: {min(labels):.2f} – {max(labels):.2f}")
else:
    print("⚠️  No label field found — will use redrob_signals as proxy labels")
    LABEL_FIELD = "__proxy__"

# Quick stats
df_meta = pd.DataFrame([{
    "yoe":   c.get("profile",{}).get("years_of_experience",0),
    "skills":len(c.get("skills",[])),
    "jobs":  len(c.get("career_history",[])),
} for c in candidates])
print("\n📊 Test set stats:")
print(df_meta.describe().round(2))

✅ Test candidates: 100,000
✅ Sample (train) candidates: 50
📋 Sample keys: ['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals']
⚠️  No label field found — will use redrob_signals as proxy labels

📊 Test set stats:
             yoe     skills       jobs
count  100000.00  100000.00  100000.00
mean        7.17       9.60       3.00
std         3.82       3.31       1.52
min         1.00       5.00       1.00
25%         3.90       7.00       2.00
50%         6.80       9.00       3.00
75%         9.90      11.00       4.00
max        16.90      23.00       9.00


In [4]:
# ── SECTION 3: JD Decomposition ──────────────────────────────────────────────
from docx import Document

def read_docx(path):
    doc = Document(path)
    return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

jd_raw = read_docx(JD_FILE)

def parse_jd(jd_text):
    jd_lower = jd_text.lower()
    seniority = "mid"
    if any(w in jd_lower for w in ["senior","lead","principal","staff"]):
        seniority = "senior"
    elif any(w in jd_lower for w in ["junior","entry","fresher"]):
        seniority = "junior"
    yoe_match = re.findall(r'(\d+)\+?\s*years?\s*(?:of)?\s*experience', jd_lower)
    min_years = int(yoe_match[0]) if yoe_match else 0
    must_s = re.search(r'(?:required|must.have|mandatory)[:\s]+(.*?)(?=\n\n|\Z)',
                        jd_text, re.IGNORECASE|re.DOTALL)
    nice_s = re.search(r'(?:preferred|nice.to.have|bonus|plus)[:\s]+(.*?)(?=\n\n|\Z)',
                        jd_text, re.IGNORECASE|re.DOTALL)
    return {
        "raw":        jd_text,
        "seniority":  seniority,
        "min_years":  min_years,
        "must_text":  must_s.group(1) if must_s else jd_text,
        "nice_text":  nice_s.group(1) if nice_s else "",
        "full_query": jd_text,
    }

jd = parse_jd(jd_raw)
print(f"✅ Seniority: {jd['seniority']} | Min YoE: {jd['min_years']}")

# Extract JD skills
TECH_SKILLS = [
    "python","java","javascript","typescript","go","rust",
    "sql","nosql","postgresql","mongodb","redis",
    "pytorch","tensorflow","keras","scikit-learn","xgboost","lightgbm",
    "docker","kubernetes","aws","gcp","azure",
    "spark","kafka","airflow","dbt","pyspark","hadoop",
    "faiss","elasticsearch","llm","rag","transformers","bert",
    "nlp","mlops","mlflow","wandb","huggingface","langchain",
    "react","fastapi","flask","django","git","linux","bash",
]
jd_skills_must = set(s for s in TECH_SKILLS if s in jd["must_text"].lower())
jd_skills_nice = set(s for s in TECH_SKILLS if s in jd["nice_text"].lower())
print(f"Must-have skills: {jd_skills_must}")
print(f"Nice-to-have:     {jd_skills_nice}")

✅ Seniority: senior | Min YoE: 0
Must-have skills: {'xgboost', 'langchain', 'git', 'nlp', 'rag', 'rust', 'faiss', 'python', 'elasticsearch', 'go', 'transformers', 'llm'}
Nice-to-have:     {'llm', 'rag', 'go'}


In [5]:
# ── SECTION 4: Feature Engineering Functions (shared train+test) ──────────────

TODAY = datetime.now()
BUILDER_VERBS = {
    "built","deployed","designed","developed","architected",
    "implemented","shipped","launched","led","created",
    "engineered","scaled","optimized","migrated","automated",
    "reduced","increased","improved","delivered","established"
}
PASSIVE_VERBS = {"familiar","exposure","knowledge","understanding",
                 "worked","used","assisted","supported","helped"}
TITLE_RANK = {
    "intern":0,"junior":1,"associate":1,"analyst":2,"engineer":2,
    "developer":2,"senior":3,"staff":4,"lead":4,"principal":5,
    "manager":4,"director":5,"architect":4,"vp":6,"cto":7,"head":5
}
PROF_WEIGHT = {"advanced":1.0,"intermediate":0.7,"beginner":0.3}

def parse_date_str(s):
    if not s or str(s).lower() in ("null","none","present","current"):
        return TODAY
    for fmt in ("%Y-%m-%d","%Y-%m","%Y"):
        try: return datetime.strptime(str(s)[:10], fmt)
        except: pass
    return TODAY

def extract_features(c, cos_score=0.0, rrf_score=0.0,
                      bm25_score=0.0, reranker_score=0.0):
    """
    Master feature function — used for BOTH training (50 samples)
    and inference (100k candidates). Never let these diverge.
    """
    profile = c.get("profile", {}) or {}
    career  = c.get("career_history", []) or []
    skills  = c.get("skills", []) or []
    sig     = c.get("redrob_signals", {}) or {}
    edu     = c.get("education", []) or []

    # ── Builder signals ───────────────────────────────────────────────────────
    all_desc = " ".join([(j.get("description","") or "") for j in career]).lower()
    words    = all_desc.split()
    total    = max(len(words), 1)
    builder_c = sum(1 for w in words if w.rstrip(".,;") in BUILDER_VERBS)
    passive_c = sum(1 for w in words if w.rstrip(".,;") in PASSIVE_VERBS)
    impact_c  = len(re.findall(
        r'\d+(?:\.\d+)?[%kKmMbB]|\d+(?:\.\d+)?\s*(?:million|billion|percent)',all_desc))

    # ── Skill signals ─────────────────────────────────────────────────────────
    skill_map = {}
    for s in skills:
        n = (s.get("name","") or "").lower().strip()
        if n: skill_map[n] = s
    must_covered = jd_skills_must & set(skill_map)
    nice_covered = jd_skills_nice & set(skill_map)
    w_cov = sum(PROF_WEIGHT.get(skill_map[s].get("proficiency",""),0.5)
                for s in must_covered) / max(len(jd_skills_must),1)
    adv_count    = sum(1 for s in skills if s.get("proficiency")=="advanced")
    avg_dur      = np.mean([s.get("duration_months",0) for s in skills]) if skills else 0
    total_end    = sum(s.get("endorsements",0) for s in skills)
    match_end    = sum(skill_map[s].get("endorsements",0) for s in must_covered)

    # ── Trajectory signals ────────────────────────────────────────────────────
    total_yoe = profile.get("years_of_experience", 0) or 0
    timeline  = []
    for job in career:
        start = parse_date_str(job.get("start_date"))
        end   = parse_date_str(job.get("end_date"))
        dur   = job.get("duration_months",0) or 0
        rank  = next((v for k,v in TITLE_RANK.items()
                      if k in (job.get("title","") or "").lower()), 2)
        if start: timeline.append({"end":end or TODAY,"rank":rank,
                                    "months":dur,"cur":job.get("is_current",False)})
    ranks   = [t["rank"] for t in timeline]
    slope   = (ranks[-1]-ranks[0])/max(len(ranks)-1,1) if ranks else 0
    recency = sum(t["months"]*math.exp(-0.05*max((TODAY-t["end"]).days/30,0))
               for t in timeline)
    is_emp  = int(any(t["cur"] for t in timeline))
    longest = max((t["months"] for t in timeline), default=0)
    last_a  = parse_date_str(sig.get("last_active_date"))
    days_act= (TODAY-last_a).days if last_a else 999

    # ── redrob_signals ────────────────────────────────────────────────────────
    assess_raw  = sig.get("skill_assessment_scores", {}) or {}
    assess_vals = [v for v in assess_raw.values()
                   if isinstance(v, (int,float))] if isinstance(assess_raw,dict) else []
    avg_assess  = float(np.mean(assess_vals)) if assess_vals else 0.0

    # ── Education signals ─────────────────────────────────────────────────────
    tier_map    = {"tier_1":3,"tier_2":2,"tier_3":1}
    best_tier   = max((tier_map.get(e.get("tier",""),0) for e in edu), default=0)

    return {
        # Retrieval (0 during training, filled during inference)
        "cos_score":              cos_score,
        "rrf_score":              rrf_score,
        "bm25_score":             bm25_score,
        "reranker_score":         reranker_score,
        # Builder
        "builder_verb_density":   builder_c / total,
        "builder_passive_ratio":  builder_c / max(passive_c,1),
        "impact_count":           impact_c,
        "advanced_skill_count":   adv_count,
        "avg_skill_duration":     avg_dur,
        "total_endorsements":     total_end,
        "matched_endorsements":   match_end,
        # Skill gap
        "must_coverage":          len(must_covered)/max(len(jd_skills_must),1),
        "weighted_must_coverage":  w_cov,
        "nice_coverage":          len(nice_covered)/max(len(jd_skills_nice),1),
        "total_skills_count":     len(skill_map),
        # Trajectory
        "total_yoe":              total_yoe,
        "trajectory_slope":       slope,
        "recency_score":          recency,
        "is_currently_employed":  is_emp,
        "longest_tenure_months":  longest,
        "days_since_active":      days_act,
        # redrob_signals
        "github_activity_score":  sig.get("github_activity_score",0) or 0,
        "profile_completeness":   sig.get("profile_completeness_score",0) or 0,
        "avg_assessment_score":   avg_assess,
        "interview_rate":         sig.get("interview_completion_rate",0) or 0,
        "offer_rate":             sig.get("offer_acceptance_rate",0) or 0,
        "open_to_work":           int(sig.get("open_to_work_flag",False)),
        # Seniority
        "seniority_match": 1.0 if (
            (jd["seniority"]=="senior" and total_yoe>=5) or
            (jd["seniority"]=="junior" and total_yoe<=3) or
            (jd["seniority"]=="mid")) else 0.0,
        "yoe_meets_min":   1.0 if total_yoe >= jd.get("min_years",0) else 0.0,
        "education_tier":  best_tier,
    }

FEATURE_COLS = list(extract_features(candidates[0]).keys())
print(f"✅ Feature count: {len(FEATURE_COLS)}")
print(f"   Features: {FEATURE_COLS}")

✅ Feature count: 30
   Features: ['cos_score', 'rrf_score', 'bm25_score', 'reranker_score', 'builder_verb_density', 'builder_passive_ratio', 'impact_count', 'advanced_skill_count', 'avg_skill_duration', 'total_endorsements', 'matched_endorsements', 'must_coverage', 'weighted_must_coverage', 'nice_coverage', 'total_skills_count', 'total_yoe', 'trajectory_slope', 'recency_score', 'is_currently_employed', 'longest_tenure_months', 'days_since_active', 'github_activity_score', 'profile_completeness', 'avg_assessment_score', 'interview_rate', 'offer_rate', 'open_to_work', 'seniority_match', 'yoe_meets_min', 'education_tier']


In [6]:
# ── SECTION 5: Train LightGBM on 50 Sample Candidates ────────────────────────
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler

# ── Step 1: Extract features for all 50 samples ───────────────────────────────
print("Extracting features from 50 sample candidates...")
train_rows = []
for c in tqdm(sample_candidates, desc="Train features"):
    # No retrieval scores during training — set to 0
    feats = extract_features(c, cos_score=0.0, rrf_score=0.0,
                              bm25_score=0.0, reranker_score=0.0)
    train_rows.append(feats)

df_train = pd.DataFrame(train_rows)

# ── Step 2: Get labels ────────────────────────────────────────────────────────
if LABEL_FIELD != "__proxy__":
    # Use provided relevance scores
    y_train = np.array([float(c.get(LABEL_FIELD, 0)) for c in sample_candidates])
    print(f"✅ Using label field: '{LABEL_FIELD}'")
else:
    # No labels → build proxy from redrob_signals + skill coverage
    print("Building proxy labels from redrob_signals...")
    proxy = (
        df_train["weighted_must_coverage"] * 0.30 +
        df_train["avg_assessment_score"].clip(0,100) / 100 * 0.25 +
        df_train["github_activity_score"].clip(0,10) / 10 * 0.20 +
        df_train["builder_passive_ratio"].clip(0,5) / 5 * 0.15 +
        df_train["profile_completeness"].clip(0,100) / 100 * 0.10
    )
    y_train = proxy.values
    print(f"✅ Proxy labels: min={y_train.min():.3f} max={y_train.max():.3f}")

# Normalise labels to [0, 1]
y_min, y_max = y_train.min(), y_train.max()
if y_max > y_min:
    y_train = (y_train - y_min) / (y_max - y_min)

# ── Step 3: Train LightGBM ────────────────────────────────────────────────────
# Use non-retrieval features only for training
# (retrieval scores are 0 in training, filled in test)
NON_RETRIEVAL_FEATS = [c for c in FEATURE_COLS
                       if c not in ("cos_score","rrf_score",
                                     "bm25_score","reranker_score")]

X_train = df_train[NON_RETRIEVAL_FEATS].fillna(0).values

lgb_params = {
    "objective":     "regression",
    "metric":        "rmse",
    "n_estimators":  200,
    "learning_rate": 0.05,
    "max_depth":     4,        # shallow — avoids overfitting on 50 samples
    "num_leaves":    15,
    "min_child_samples": 3,   # critical for small dataset
    "subsample":     0.8,
    "colsample_bytree": 0.8,
    "reg_alpha":     0.1,
    "reg_lambda":    0.1,
    "random_state":  SEED,
    "verbose":       -1,
}

lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(X_train, y_train)
print("✅ LightGBM trained on 50 samples")

# Feature importance
feat_imp = pd.DataFrame({
    "feature":    NON_RETRIEVAL_FEATS,
    "importance": lgb_model.feature_importances_
}).sort_values("importance", ascending=False)
print("\n📊 Top 10 most important features:")
print(feat_imp.head(10).to_string(index=False))

Extracting features from 50 sample candidates...


Train features:   0%|          | 0/50 [00:00<?, ?it/s]

Building proxy labels from redrob_signals...
✅ Proxy labels: min=0.032 max=0.565
✅ LightGBM trained on 50 samples

📊 Top 10 most important features:
               feature  importance
 github_activity_score         140
  avg_assessment_score         130
 builder_passive_ratio         108
  builder_verb_density          96
  profile_completeness          69
    avg_skill_duration          64
weighted_must_coverage          51
        interview_rate          45
 longest_tenure_months          43
     days_since_active          34


In [7]:
# ── SECTION 6: Serialize 100k Candidates ─────────────────────────────────────
def serialize_for_embedding(c):
    profile = c.get("profile", {})
    parts   = []
    headline = profile.get("headline", "")
    summary  = profile.get("summary", "")
    if headline: parts.append(f"Role: {headline}")
    if summary:  parts.append(f"Summary: {summary[:300]}")
    skills = c.get("skills", [])
    skill_strs = [f"{s.get('name','')} ({s.get('proficiency','')})"
                  for s in skills if s.get("name")]
    if skill_strs: parts.append("Skills: " + ", ".join(skill_strs))
    for job in c.get("career_history", []):
        desc = (job.get("description","") or "")[:200]
        parts.append(f"Experience: {job.get('title','')} at {job.get('company','')}. {desc}")
    for edu in c.get("education", []):
        parts.append(f"Education: {edu.get('degree','')} in {edu.get('field_of_study','')} from {edu.get('institution','')}")
    return " | ".join(parts)

def serialize_for_bm25(c):
    base = serialize_for_embedding(c)
    names = [s.get("name","") for s in c.get("skills",[]) if s.get("name")]
    return base + " " + " ".join(names * 3)

corpus_emb  = [serialize_for_embedding(c) for c in tqdm(candidates, desc="Serializing")]
corpus_bm25 = [serialize_for_bm25(c).split() for c in candidates]
print(f"✅ Serialized {len(corpus_emb):,} candidates")

Serializing:   0%|          | 0/100000 [00:00<?, ?it/s]

✅ Serialized 100,000 candidates


In [8]:
# ── SECTION 7: BM25 Sparse Retrieval ─────────────────────────────────────────
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(corpus_bm25)
print("✅ BM25 index built")

def bm25_retrieve(query_text, top_k=TOP_K_RETRIEVAL):
    scores  = bm25.get_scores(query_text.lower().split())
    max_s   = scores.max()
    if max_s > 0: scores = scores / max_s
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(i, float(scores[i])) for i in top_idx]

bm25_results = bm25_retrieve(jd["full_query"])
print(f"BM25 top-5:")
for idx, score in bm25_results[:5]:
    name = candidates[idx].get("profile",{}).get("anonymized_name","?")
    print(f"  {name:30s}  {score:.4f}")

✅ BM25 index built
BM25 top-5:
  Aadhya Iyer                     1.0000
  Reyansh Chowdary                0.9243
  Advaith Pillai                  0.8584
  Kiara Mittal                    0.8443
  Aryan Goyal                     0.8353


In [9]:
# ── SECTION 8: BGE-M3 Dense Embeddings ───────────────────────────────────────
from FlagEmbedding import BGEM3FlagModel

SAVED_EMB = "/kaggle/input/redrob-embeddings/cand_embs.npy"

if os.path.exists(SAVED_EMB):
    cand_embs = np.load(SAVED_EMB)
    encoder   = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
    print(f"✅ Loaded cached embeddings: {cand_embs.shape}")
elif os.path.exists(EMB_CACHE):
    cand_embs = np.load(EMB_CACHE)
    encoder   = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
    print(f"✅ Loaded working cache: {cand_embs.shape}")
else:
    encoder = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
    print("Encoding 100k on T4 GPU (~8 min)...")
    all_embs = []
    for i in tqdm(range(0, len(corpus_emb), 256), desc="Encoding"):
        batch = corpus_emb[i:i+256]
        embs  = encoder.encode(batch, batch_size=256, max_length=512,
                               return_dense=True, return_sparse=False,
                               return_colbert_vecs=False)["dense_vecs"]
        all_embs.append(embs)
    cand_embs = np.vstack(all_embs)
    np.save(EMB_CACHE, cand_embs)
    print(f"✅ Saved: {cand_embs.shape}")

jd_emb = encoder.encode([jd["full_query"]], max_length=512, return_dense=True,
                          return_sparse=False, return_colbert_vecs=False)["dense_vecs"][0]
print(f"✅ JD embedding: {jd_emb.shape}")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

Encoding:   0%|          | 0/391 [00:00<?, ?it/s]


initial target device: 100%|██████████| 2/2 [00:19<00:00,  9.69s/it]

Chunks: 100%|██████████| 2/2 [00:05<00:00,  2.94s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.24s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.25s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.27s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.28s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.29s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.32s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.33s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.34s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.38s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.40s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.39s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.42s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.43s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.48s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.48s/it]

Chunks: 100%|██████████| 2/2 [00:04<00:00,  2.49s

✅ Saved: (100000, 1024)


Chunks: 100%|██████████| 1/1 [00:00<00:00, 14.13it/s]

✅ JD embedding: (1024,)


In [10]:
# ── SECTION 9: Hybrid Retrieval → Top 300 ────────────────────────────────────
from sklearn.preprocessing import normalize

cand_embs_norm = normalize(cand_embs, norm="l2")
jd_emb_norm    = normalize(jd_emb.reshape(1,-1), norm="l2")[0]
cos_scores     = cand_embs_norm @ jd_emb_norm

emb_top_idx = np.argsort(cos_scores)[::-1][:TOP_K_RETRIEVAL]

def rrf(bm25_hits, emb_hits, k=60):
    scores = {}
    for rank, (idx, _) in enumerate(bm25_hits):
        scores[idx] = scores.get(idx,0) + 1/(k+rank+1)
    for rank, idx in enumerate(emb_hits):
        scores[idx] = scores.get(idx,0) + 1/(k+rank+1)
    return sorted(scores.items(), key=lambda x:x[1], reverse=True)

hybrid_hits       = rrf(bm25_results, emb_top_idx)[:TOP_K_RETRIEVAL]
top300_indices    = [idx for idx,_ in hybrid_hits]
top300_rrf_scores = {idx:s for idx,s in hybrid_hits}

print(f"✅ Stage 1 complete → {len(top300_indices)} candidates shortlisted")

✅ Stage 1 complete → 300 candidates shortlisted


In [11]:
# ── SECTION 10: Feature Engineering on Top 300 ───────────────────────────────
print("Extracting features for Top 300 candidates...")

feature_rows = []
for idx in tqdm(top300_indices, desc="Features"):
    c     = candidates[idx]
    bm25s = next((s for i,s in bm25_results if i==idx), 0.0)
    feats = extract_features(
        c,
        cos_score   = float(cos_scores[idx]),
        rrf_score   = top300_rrf_scores.get(idx, 0.0),
        bm25_score  = bm25s,
        reranker_score = 0.0   # filled in Section 11
    )
    feature_rows.append(feats)

df_feat = pd.DataFrame(feature_rows, index=top300_indices)
print(f"✅ Feature matrix: {df_feat.shape}")
print(df_feat[["weighted_must_coverage","avg_assessment_score",
               "github_activity_score","cos_score"]].describe().round(3))

Extracting features for Top 300 candidates...


Features:   0%|          | 0/300 [00:00<?, ?it/s]

✅ Feature matrix: (300, 30)
       weighted_must_coverage  avg_assessment_score  github_activity_score  \
count                 300.000               300.000                300.000   
mean                    0.047                38.389                 20.460   
std                     0.058                31.073                 26.143   
min                     0.000                 0.000                 -1.000   
25%                     0.000                 0.000                 -1.000   
50%                     0.025                48.417                  5.100   
75%                     0.083                64.375                 39.300   
max                     0.250                93.900                 96.900   

       cos_score  
count    300.000  
mean       0.651  
std        0.045  
min        0.512  
25%        0.611  
50%        0.678  
75%        0.683  
max        0.723  


In [13]:
# ── SECTION 11: Cross-Encoder Reranking (Manual GPU) ─────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification

RERANKER_MODEL = 'BAAI/bge-reranker-v2-m3'
device         = torch.device("cuda:0")

tokenizer_r      = AutoTokenizer.from_pretrained(RERANKER_MODEL)
reranker_model   = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL, torch_dtype=torch.float16).to(device).eval()
print(f"✅ Reranker on {next(reranker_model.parameters()).device}")

def format_pair(candidate):
    skills = [s.get("name","") for s in candidate.get("skills",[])[:20] if s.get("name")]
    exps   = [f"{j.get('title','')} at {j.get('company','')}: {(j.get('description','') or '')[:150]}"
               for j in candidate.get("career_history",[])[:3]]
    p      = candidate.get("profile",{})
    cand_str = (
        f"Headline: {p.get('headline','')}\n"
        f"Summary: {p.get('summary','')[:200]}\n"
        f"Skills: {', '.join(skills)}\n"
        f"Experience: {' | '.join(exps)}"
    )
    return [jd["full_query"][:512], cand_str]

def compute_scores_gpu(pairs, batch_size=32):
    all_scores = []
    with torch.no_grad():
        for i in tqdm(range(0, len(pairs), batch_size), desc="Reranking"):
            batch   = pairs[i:i+batch_size]
            encoded = tokenizer_r(
                [p[0] for p in batch],
                [p[1] for p in batch],
                padding=True, truncation=True,
                max_length=512, return_tensors="pt"
            ).to(device)
            logits = reranker_model(**encoded).logits.squeeze(-1)
            scores = torch.sigmoid(logits).cpu().float().tolist()
            if isinstance(scores, float): scores = [scores]
            all_scores.extend(scores)
    return all_scores

# Score Top-150 from Stage 1
top150_for_rerank = top300_indices[:TOP_K_RERANK]
pairs             = [format_pair(candidates[i]) for i in top150_for_rerank]

reranker_scores   = compute_scores_gpu(pairs, batch_size=32)
reranker_score_map= {idx:s for idx,s in zip(top150_for_rerank, reranker_scores)}

# Update reranker_score in feature matrix
for idx in top150_for_rerank:
    if idx in df_feat.index:
        df_feat.loc[idx, "reranker_score"] = reranker_score_map[idx]

print(f"✅ Reranked {len(top150_for_rerank)} candidates")
print(f"Score range: {min(reranker_scores):.3f} – {max(reranker_scores):.3f}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker on cuda:0


Reranking:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Reranked 200 candidates
Score range: 0.006 – 0.989


In [14]:
# ── SECTION 12: LightGBM Inference → Top 100 ─────────────────────────────────

# ── Step 1: LightGBM scores non-retrieval features ───────────────────────────
X_test      = df_feat[NON_RETRIEVAL_FEATS].fillna(0).values
lgb_scores  = lgb_model.predict(X_test)
df_feat["lgb_score"] = lgb_scores

# ── Step 2: Fuse LightGBM + retrieval scores ─────────────────────────────────
scaler    = MinMaxScaler()
ALL_SCORE_COLS = ["lgb_score","reranker_score","cos_score",
                  "rrf_score","bm25_score"]
df_scores = pd.DataFrame(
    scaler.fit_transform(df_feat[ALL_SCORE_COLS].fillna(0)),
    columns=ALL_SCORE_COLS, index=df_feat.index
)

# Fusion weights — LightGBM gets highest weight (learned from labelled data)
FUSION_WEIGHTS = {
    "lgb_score":      0.35,   # trained on 50 samples ← most trusted
    "reranker_score": 0.30,   # cross-encoder semantic precision
    "cos_score":      0.15,   # BGE-M3 semantic similarity
    "rrf_score":      0.12,   # hybrid retrieval rank
    "bm25_score":     0.08,   # keyword match
}

df_scores["final_score"] = sum(
    df_scores[col] * w for col, w in FUSION_WEIGHTS.items()
)

# ── Step 3: Select Top 100 ────────────────────────────────────────────────────
top100_df      = df_scores.sort_values("final_score", ascending=False).head(TOP_K_FINAL)
top100_indices = top100_df.index.tolist()

print(f"✅ Final Top-{TOP_K_FINAL} selected")
print("\nTop 10 preview:")
for rank, idx in enumerate(top100_indices[:10], 1):
    c    = candidates[idx]
    name = c.get("profile",{}).get("anonymized_name","?")
    score= top100_df.loc[idx,"final_score"]
    print(f"  #{rank:<3d} {name:30s}  {score:.4f}")

✅ Final Top-100 selected

Top 10 preview:
  #1   Aryan Goyal                     0.9346
  #2   Sai Verma                       0.9296
  #3   Aditya Pillai                   0.9107
  #4   Arnav Ghosh                     0.9058
  #5   Tanvi Mukherjee                 0.8986
  #6   Advaith Pillai                  0.8948
  #7   Aarohi Bose                     0.8868
  #8   Kiara Mittal                    0.8788
  #9   Saanvi Naidu                    0.8764
  #10  Aditya Subramanian              0.8722


In [15]:
# ── SECTION 13: Explanation Cards (Top 10 preview) ───────────────────────────
def make_card(idx, rank):
    c       = candidates[idx]
    profile = c.get("profile",{})
    sig     = c.get("redrob_signals",{}) or {}
    feats   = df_feat.loc[idx]
    gap_covered = [s for s in jd_skills_must
                   if s in set((sk.get("name","") or "").lower()for sk in c.get("skills",[]))]
    gap_missing = [s for s in jd_skills_must if s not in gap_covered]

    name  = profile.get("anonymized_name","?")
    title = profile.get("current_title","")
    pct   = int(top100_df.loc[idx,"final_score"]*100)

    signals = []
    if feats["builder_passive_ratio"] > 2:
        signals.append(("✅","Builder profile — strong action verbs"))
    else:
        signals.append(("⚠️","Passive language in experience"))
    if feats["impact_count"] >= 3:
        signals.append(("✅",f"Quantified impact ({int(feats['impact_count'])} metrics)"))
    if feats["total_yoe"] >= jd.get("min_years",0):
        signals.append(("✅",f"{feats['total_yoe']:.1f} yrs (meets minimum)"))
    else:
        signals.append(("❌",f"{feats['total_yoe']:.1f} yrs (below minimum)"))
    if gap_covered:
        signals.append(("✅",f"Core skills: {', '.join(gap_covered[:4])}"))
    if gap_missing:
        signals.append(("⚠️",f"Missing: {', '.join(gap_missing[:3])}"))
    if feats["avg_assessment_score"] > 60:
        signals.append(("✅",f"Assessment: {feats['avg_assessment_score']:.0f}/100"))
    if feats["github_activity_score"] > 5:
        signals.append(("✅",f"GitHub: {feats['github_activity_score']}/10"))
    if sig.get("open_to_work_flag"):
        signals.append(("✅","Open to work"))

    sep = "─"*60
    print(f"\n{sep}")
    print(f"#{rank:<3d} {name:<28s} [{title}]  {pct}%")
    print(sep)
    for icon, msg in signals:
        print(f"  {icon}  {msg}")
    print(sep)
    return {"rank":rank,"name":name,"score_pct":pct,"signals":signals,
            "skills_match":gap_covered,"skills_missing":gap_missing}

print("=== TOP 10 EXPLANATION CARDS ===")
[make_card(idx, rank) for rank, idx in enumerate(top100_indices[:10], 1)]

=== TOP 10 EXPLANATION CARDS ===

────────────────────────────────────────────────────────────
#1   Aryan Goyal                  [Senior AI Engineer]  93%
────────────────────────────────────────────────────────────
  ✅  Builder profile — strong action verbs
  ✅  5.9 yrs (meets minimum)
  ✅  Core skills: python
  ⚠️  Missing: xgboost, langchain, git
  ✅  Assessment: 93/100
  ✅  GitHub: 58.0/10
  ✅  Open to work
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
#2   Sai Verma                    [Senior AI Engineer]  92%
────────────────────────────────────────────────────────────
  ✅  Builder profile — strong action verbs
  ✅  Quantified impact (5 metrics)
  ✅  7.8 yrs (meets minimum)
  ✅  Core skills: rag, elasticsearch
  ⚠️  Missing: xgboost, langchain, git
  ✅  Assessment: 80/100
  ✅  GitHub: 82.8/10
  ✅  Open to work
────────────────────────────────────────────────────────────

─────────────────────────────────

[{'rank': 1,
  'name': 'Aryan Goyal',
  'score_pct': 93,
  'signals': [('✅', 'Builder profile — strong action verbs'),
   ('✅', '5.9 yrs (meets minimum)'),
   ('✅', 'Core skills: python'),
   ('⚠️', 'Missing: xgboost, langchain, git'),
   ('✅', 'Assessment: 93/100'),
   ('✅', 'GitHub: 58.0/10'),
   ('✅', 'Open to work')],
  'skills_match': ['python'],
  'skills_missing': ['xgboost',
   'langchain',
   'git',
   'nlp',
   'rag',
   'rust',
   'faiss',
   'elasticsearch',
   'go',
   'transformers',
   'llm']},
 {'rank': 2,
  'name': 'Sai Verma',
  'score_pct': 92,
  'signals': [('✅', 'Builder profile — strong action verbs'),
   ('✅', 'Quantified impact (5 metrics)'),
   ('✅', '7.8 yrs (meets minimum)'),
   ('✅', 'Core skills: rag, elasticsearch'),
   ('⚠️', 'Missing: xgboost, langchain, git'),
   ('✅', 'Assessment: 80/100'),
   ('✅', 'GitHub: 82.8/10'),
   ('✅', 'Open to work')],
  'skills_match': ['rag', 'elasticsearch'],
  'skills_missing': ['xgboost',
   'langchain',
   'git',
   'nl

In [16]:
# ── SECTION 14: Sanity Check ──────────────────────────────────────────────────
print("=== Top-100 vs Random-100 ===")

random_idx   = np.random.choice(len(candidates), 100, replace=False).tolist()
random_feats = pd.DataFrame([extract_features(candidates[i]) for i in random_idx])

checks = [
    ("weighted_must_coverage", "Must-skill coverage"),
    ("avg_assessment_score",   "Assessment score"),
    ("github_activity_score",  "GitHub score"),
    ("builder_passive_ratio",  "Builder ratio"),
    ("impact_count",           "Impact count"),
    ("total_yoe",              "Years of experience"),
]

print(f"\n{'Metric':<30s} {'Top-100':>9s} {'Random':>9s} {'Delta':>9s}")
print("─"*60)
for col, label in checks:
    t = df_feat.loc[top100_indices, col].mean()
    r = random_feats[col].mean()
    d = t - r
    print(f"{'✅' if d>0 else '❌'} {label:<28s} {t:>9.3f} {r:>9.3f} {d:>+9.3f}")
print("─"*60)

=== Top-100 vs Random-100 ===

Metric                           Top-100    Random     Delta
────────────────────────────────────────────────────────────
✅ Must-skill coverage              0.061     0.031    +0.030
✅ Assessment score                62.105    11.866   +50.239
✅ GitHub score                    42.750    10.659   +32.091
✅ Builder ratio                    3.349     1.967    +1.383
✅ Impact count                     2.000     0.830    +1.170
❌ Years of experience              6.657     7.389    -0.732
────────────────────────────────────────────────────────────


In [18]:
# ── SECTION 15: Export & Download ────────────────────────────────────────────
import yaml

# ── 1. Build submission dataframe ─────────────────────────────────────────────
rows = []
for rank, idx in enumerate(top100_indices, 1):
    c = candidates[idx]
    rows.append({
        "rank":         rank,
        "candidate_id": c.get("candidate_id"),
        "name":         c.get("profile",{}).get("anonymized_name"),
        "score":        round(top100_df.loc[idx,"final_score"], 6),
    })
df_sub = pd.DataFrame(rows)

# ── 2. Save CSV ───────────────────────────────────────────────────────────────
CSV_PATH = f"{OUTPUT_DIR}/submission.csv"
df_sub.to_csv(CSV_PATH, index=False)
print(f"✅ Saved: {CSV_PATH}")
print(f"   Rows: {len(df_sub)} candidates")

# ── 3. Print full CSV for copy-paste download ─────────────────────────────────
print("\n" + "="*60)
print("COPY EVERYTHING BELOW THIS LINE INTO submission.csv")
print("="*60)
print(df_sub.to_csv(index=False))
print("="*60)

# ── 4. Detailed JSON ──────────────────────────────────────────────────────────
detailed = []
for rank, idx in enumerate(top100_indices, 1):
    c   = candidates[idx]
    sig = c.get("redrob_signals",{}) or {}
    f   = df_feat.loc[idx]
    assess_raw  = sig.get("skill_assessment_scores",{}) or {}
    assess_vals = [v for v in assess_raw.values()
                   if isinstance(v,(int,float))] if isinstance(assess_raw,dict) else []
    detailed.append({
        "rank":           rank,
        "candidate_id":   c.get("candidate_id"),
        "name":           c.get("profile",{}).get("anonymized_name"),
        "current_title":  c.get("profile",{}).get("current_title"),
        "match_score_pct":round(top100_df.loc[idx,"final_score"]*100,1),
        "total_yoe":      round(float(f["total_yoe"]),1),
        "lgb_score":      round(float(f["lgb_score"]),4),
        "reranker_score": round(float(f["reranker_score"]),4),
        "assessment_score":round(float(np.mean(assess_vals)) if assess_vals else 0,1),
        "github_score":   sig.get("github_activity_score",0),
        "open_to_work":   sig.get("open_to_work_flag",False),
    })

JSON_PATH = f"{OUTPUT_DIR}/ranked_candidates_detailed.json"
with open(JSON_PATH,"w") as fp:
    json.dump(detailed, fp, indent=2)
print(f"✅ Saved: {JSON_PATH}")

# ── 5. Metadata YAML ──────────────────────────────────────────────────────────
meta = {
    "model":         "BGE-M3 + BM25 + BGE-Reranker + LightGBM",
    "train_samples": len(sample_candidates),
    "test_samples":  len(candidates),
    "top_k_final":   TOP_K_FINAL,
    "features":      FEATURE_COLS,
    "generated_at":  datetime.now().isoformat(),
}
YAML_PATH = f"{OUTPUT_DIR}/submission_metadata.yaml"
with open(YAML_PATH,"w") as fp:
    yaml.dump(meta, fp)
print(f"✅ Saved: {YAML_PATH}")

# ── 6. Validate ───────────────────────────────────────────────────────────────
if os.path.exists(VALIDATE_PY):
    !python {VALIDATE_PY} --submission {CSV_PATH}
else:
    print("ℹ️  Validator not found — verify manually")

print("\n🎉 Done! Top 100 candidates ranked and exported.")
print("📋 Copy the CSV printed above to submit.")

✅ Saved: /kaggle/working/submission.csv
   Rows: 100 candidates

COPY EVERYTHING BELOW THIS LINE INTO submission.csv
rank,candidate_id,name,score
1,CAND_0005538,Aryan Goyal,0.934583
2,CAND_0071974,Sai Verma,0.929597
3,CAND_0094759,Aditya Pillai,0.910688
4,CAND_0093547,Arnav Ghosh,0.905784
5,CAND_0046525,Tanvi Mukherjee,0.898584
6,CAND_0061257,Advaith Pillai,0.894816
7,CAND_0093193,Aarohi Bose,0.886808
8,CAND_0080766,Kiara Mittal,0.878821
9,CAND_0046064,Saanvi Naidu,0.876447
10,CAND_0006567,Aditya Subramanian,0.872224
11,CAND_0033861,Kabir Kapoor,0.870389
12,CAND_0077337,Aarav Agarwal,0.859375
13,CAND_0039754,Mira Banerjee,0.849137
14,CAND_0007411,Rahul Bansal,0.83511
15,CAND_0088025,Amit Arora,0.827023
16,CAND_0008425,Myra Krishnan,0.812374
17,CAND_0018499,Aarav Trivedi,0.808177
18,CAND_0081846,Arjun Khanna,0.800963
19,CAND_0002025,Ira Dalal,0.799467
20,CAND_0011687,Shreya Tiwari,0.79856
21,CAND_0037980,Reyansh Chowdary,0.798175
22,CAND_0030468,Pooja Bose,0.782335
23,CAND_0055905,Anika